# UPE Data – Exploratory Data Analysis

This notebook contains two parts:

| Part | Focus |
|------|-------|
| **A – Time-series analysis** | How do photon counts evolve over time? |
| **B – Count distribution analysis** | What is the statistical shape of the photon count distribution? |

In [ ]:
"""
UPE Master Analysis Script --> ALLE CODES TOT NU TOE SAMENGEVOEGD
Combineert Kanaal-asymmetrie (Palmair/Dorsaal), Demografie (Man/Vrouw) 
en de Cumulatieve Area Under the Curve (AUC) in één interactief dashboard.
"""

import os
import warnings
import re

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# IPython & ipywidgets imports
from IPython.display import display
from IPython.display import HTML as DisplayHTML 
from ipywidgets import (
    HBox, HTML, Checkbox, IntSlider, Layout, VBox, fixed, interactive_output,
)

# ---------------------------------------------------------
# Configuration
# ---------------------------------------------------------
# Controleer of dit het juiste pad is naar je data!
DATA_FOLDER = r'/Users/jasperslot/Documents/Desktop - Jasper’s MacBook Air (2)/N&S jaar 1/BLOK 6/UPE_Project/Echte_data' 
FILE_EXTENSION = '.txt'
ALL_CHANNELS = ['LD', 'LP', 'RP', 'RD']

LEFT_CHANNELS = ('LD', 'LP')
RIGHT_CHANNELS = ('RP', 'RD')

LEFT_KEYWORDS = ('links', 'left')
RIGHT_KEYWORDS = ('rechts', 'right')

BG_HEADER_SIGNALS = ('Photon counts - Left', 'Photon counts - Right')
BG_USECOLS = [1, 3]
BG_COL_NAMES = ['BG_Left', 'BG_Right']

BIN_ROWS = 20 

PHASE_BASELINE_END = 120  
PHASE_INTERVENTION_END = 300  
PHASE_RECOVERY_END = 600  

SEX_LOOKUP = {
    "p1" : "male", "p2" : "female", "p3" : "male", "p4" : "female",
    "p5" : "male", "p6" : "male", "p7" : "male", "p8" : "male",
    "p9" : "female", "p10" : "male", "p11" : "male", "p12" : "female",
    "p13" : "female", "p14" : "female", "p15" : "male",
}

# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------
def _is_background_file(file_path: str) -> bool:
    try:
        with open(file_path, encoding='utf-8', errors='replace') as f:
            header = f.readline()
        return any(sig in header for sig in BG_HEADER_SIGNALS)
    except OSError:
        return False

def _infer_channels(file_name: str, file_path: str) -> list[str] | None:
    name = file_name.lower()
    if any(k in name for k in LEFT_KEYWORDS): return list(LEFT_CHANNELS)
    if any(k in name for k in RIGHT_KEYWORDS): return list(RIGHT_CHANNELS)
    try:
        with open(file_path, encoding='utf-8', errors='replace') as f:
            header = f.readline()
        if any(ch in header for ch in LEFT_CHANNELS): return list(LEFT_CHANNELS)
        if any(ch in header for ch in RIGHT_CHANNELS): return list(RIGHT_CHANNELS)
    except OSError as exc:
        warnings.warn(f"Could not read header of '{file_name}': {exc}")
    return None

def extract_participant(file_name: str) -> str | None:
    match = re.search(r"exercise_(p\d+)_", file_name.lower())
    if match: return match.group(1)
    return None

def _group_avg(all_df: pd.DataFrame, grouper) -> pd.DataFrame:
    avg = all_df.T.groupby(grouper).mean().T
    avg.columns = [f"{c}_Avg" for c in avg.columns]
    return avg.dropna(axis=1, how='all')

def _bin_to_cps(data: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    n = (len(data) // BIN_ROWS) * BIN_ROWS
    binned = data.iloc[:n].groupby(np.arange(n) // BIN_ROWS).sum().reset_index(drop=True)
    return binned, pd.Series(np.arange(len(binned)), dtype=float)

def _safe_column_concat(frames: list[pd.DataFrame], keys: list[str]) -> pd.DataFrame:
    max_len = max(len(f) for f in frames)
    padded = [f.reindex(range(max_len)) for f in frames]
    return pd.concat(padded, keys=keys, axis=1)

def calc_robust_pct(a, b):
    a = pd.Series(a)
    b = pd.Series(b)
    b = b.replace(0, np.nan)
    return ((a - b) / b) * 100

def _build_average_frames(all_df: pd.DataFrame) -> tuple[list[pd.DataFrame], list[str], dict]:
    avg_per_file = all_df.T.groupby(level=0).mean().T
    overall_avg = avg_per_file.mean(axis=1).rename('Overall_Avg')

    sex_level = all_df.columns.get_level_values("sex")
    channel_level = all_df.columns.get_level_values("channel")
    hand_grouper = ['Left' if ch in LEFT_CHANNELS else 'Right' for ch in channel_level]
    
    sex_avg = _group_avg(all_df, sex_level)
    hand_avg = _group_avg(all_df, hand_grouper)
    channel_avg = _group_avg(all_df, channel_level)

    # Bereken gemiddeldes specifiek voor Man/Vrouw per kanaal
    sex_channel_frames = {}
    for sex in ["male", "female"]:
        for channel in ["LP", "RP", "LD", "RD"]:
            mask = ((sex_level == sex) & (channel_level == channel))
            col_name = f"{channel}_{sex}_Avg"
            if mask.any():
                sex_channel_frames[col_name] = all_df.loc[:, mask].mean(axis=1)
            else:
                sex_channel_frames[col_name] = pd.Series(np.nan, index=all_df.index)

    sex_channel_df = pd.DataFrame(sex_channel_frames)
    
    sex_channel_df["Left_male_Avg"] = sex_channel_df[["LP_male_Avg", "LD_male_Avg"]].mean(axis=1)
    sex_channel_df["Right_male_Avg"] = sex_channel_df[["RP_male_Avg", "RD_male_Avg"]].mean(axis=1)
    sex_channel_df["Palmar_male_Avg"] = sex_channel_df[["LP_male_Avg", "RP_male_Avg"]].mean(axis=1)
    sex_channel_df["Dorsal_male_Avg"] = sex_channel_df[["LD_male_Avg", "RD_male_Avg"]].mean(axis=1)

    sex_channel_df["Left_female_Avg"] = sex_channel_df[["LP_female_Avg", "LD_female_Avg"]].mean(axis=1)
    sex_channel_df["Right_female_Avg"] = sex_channel_df[["RP_female_Avg", "RD_female_Avg"]].mean(axis=1)
    sex_channel_df["Palmar_female_Avg"] = sex_channel_df[["LP_female_Avg", "RP_female_Avg"]].mean(axis=1)
    sex_channel_df["Dorsal_female_Avg"] = sex_channel_df[["LD_female_Avg", "RD_female_Avg"]].mean(axis=1)

    palmar_mask = channel_level.isin(['LP', 'RP'])
    dorsal_mask = channel_level.isin(['LD', 'RD'])
    palmar_avg = all_df.loc[:, palmar_mask].mean(axis=1).rename('Palmar_Avg (LP+RP)')
    dorsal_avg = all_df.loc[:, dorsal_mask].mean(axis=1).rename('Dorsal_Avg (LD+RD)')

    # Vluchtige verschillen (voor in de grafiek)
    diff_avg = (palmar_avg - dorsal_avg).rename('Verschil (Palmair - Dorsaal)')
    pct_diff_avg = calc_robust_pct(palmar_avg, dorsal_avg).rename('Procentueel_Verschil (%)')

    ld_avg = all_df.loc[:, channel_level == 'LD'].mean(axis=1) if 'LD' in channel_level else pd.Series(np.nan, index=all_df.index)
    lp_avg = all_df.loc[:, channel_level == 'LP'].mean(axis=1) if 'LP' in channel_level else pd.Series(np.nan, index=all_df.index)
    rp_avg = all_df.loc[:, channel_level == 'RP'].mean(axis=1) if 'RP' in channel_level else pd.Series(np.nan, index=all_df.index)
    rd_avg = all_df.loc[:, channel_level == 'RD'].mean(axis=1) if 'RD' in channel_level else pd.Series(np.nan, index=all_df.index)

    pct_diff_lp_rp = calc_robust_pct(lp_avg, rp_avg).rename('Procentueel_Verschil (LP vs RP) (%)')
    pct_diff_ld_rd = calc_robust_pct(ld_avg, rd_avg).rename('Procentueel_Verschil (LD vs RD) (%)')

    meas_parts = {}
    for i in range(1, 5):
        cols = [c for c in avg_per_file.columns if f'_m{i}_' in c]
        if cols: meas_parts[f'm{i}_Avg'] = avg_per_file[cols].mean(axis=1)
    meas_avg = pd.DataFrame(meas_parts)

    indiv_df = all_df.copy()
    indiv_df.columns = ["_".join(str(v) for v in col if pd.notna(v)) for col in indiv_df.columns]
    indiv_df.dropna(axis=1, how='all', inplace=True)

    sex_diff_df = pd.DataFrame(index=all_df.index)
    comparisons = {
        "LP": ("LP_male_Avg", "LP_female_Avg"), "RP": ("RP_male_Avg", "RP_female_Avg"),
        "LD": ("LD_male_Avg", "LD_female_Avg"), "RD": ("RD_male_Avg", "RD_female_Avg"),
        "Links": ("Left_male_Avg", "Left_female_Avg"), "Rechts": ("Right_male_Avg", "Right_female_Avg"),
        "Palmair": ("Palmar_male_Avg", "Palmar_female_Avg"), "Dorsaal": ("Dorsal_male_Avg", "Dorsal_female_Avg"),
    }
    for label, (male_col, female_col) in comparisons.items():
        sex_diff_df[f"{label}_MV_pct"] = calc_robust_pct(sex_channel_df[male_col], sex_channel_df[female_col])

    parts = [indiv_df, avg_per_file, overall_avg, hand_avg, sex_avg, channel_avg, sex_channel_df, 
             sex_diff_df, palmar_avg, dorsal_avg, diff_avg, pct_diff_avg, pct_diff_lp_rp, pct_diff_ld_rd]
    if not meas_avg.empty: parts.append(meas_avg)

    plot_groups = {
        'hand': list(hand_avg.columns), 'channel': list(channel_avg.columns),
        'sex': list(sex_avg.columns), 'sex_channel': list(sex_channel_df.columns),
        'sex_diff': list(sex_diff_df.columns),
    }
    return parts, list(indiv_df.columns), plot_groups

# ---------------------------------------------------------
# Data loading
# ---------------------------------------------------------
def load_and_process_data(folder_path: str):
    if not os.path.isdir(folder_path):
        print(f"Error: directory not found — '{folder_path}'")
        return pd.DataFrame(), [], {}, []

    frames, labels, bg_cols, column_metadata = [], [], {}, []

    for file_name in sorted(os.listdir(folder_path)):
        if not file_name.endswith(FILE_EXTENSION): continue
        path = os.path.join(folder_path, file_name)
        stem = os.path.splitext(file_name)[0]

        if _is_background_file(path):
            try:
                bg = pd.read_csv(path, sep="\t", skiprows=1, header=None, usecols=BG_USECOLS)
                bg.columns = BG_COL_NAMES
                bg_cols[f"{stem}_BG_Left"] = bg['BG_Left']
                bg_cols[f"{stem}_BG_Right"] = bg['BG_Right']
            except Exception as exc: warnings.warn(f"Skipping background '{file_name}': {exc}")
            continue

        channels = _infer_channels(file_name, path)
        if channels is None: continue

        try:
            df = pd.read_csv(path, sep="\t", skiprows=1, header=None, usecols=[1, 3])
            df.columns = channels
            df = df.reindex(columns=ALL_CHANNELS, fill_value=np.nan)

            participant = extract_participant(stem) or "unknown"
            sex = SEX_LOOKUP.get(participant, "unknown")

            frames.append(df)
            labels.append(stem)
            for channel in ALL_CHANNELS:
                column_metadata.append({"file": stem, "participant": participant, "sex": sex, "channel": channel})
        except Exception as exc:
            warnings.warn(f"Skipping '{file_name}': {exc}")

    if not frames and not bg_cols: return pd.DataFrame(), [], {}, []

    if frames:
        all_df = _safe_column_concat(frames, labels)
        all_df.columns = pd.MultiIndex.from_tuples(
            [(m["file"], m["participant"], m["sex"], m["channel"]) for m in column_metadata],
            names=["file", "participant", "sex", "channel"]
        )
        parts, indiv_labels, plot_groups = _build_average_frames(all_df)
        processed_df = pd.concat(parts, axis=1)
    else:
        processed_df, indiv_labels, plot_groups = pd.DataFrame(), [], {}

    if bg_cols:
        bg_frame = pd.DataFrame(bg_cols)
        max_len = max(len(processed_df), len(bg_frame)) if not processed_df.empty else len(bg_frame)
        processed_df = pd.concat([processed_df.reindex(range(max_len)), bg_frame.reindex(range(max_len))], axis=1) if not processed_df.empty else bg_frame
        indiv_labels += list(bg_cols.keys())

    processed_df.insert(0, "Time", np.arange(len(processed_df), dtype=float))
    return processed_df, labels, plot_groups, indiv_labels

# ---------------------------------------------------------
# Plotting & Stats (De Geïntegreerde Functie)
# ---------------------------------------------------------
def plot_combined_data(
    df, file_labels, plot_groups, individual_labels,
    window, counts_per_sec, demean, start_at_zero, 
    show_all_individual, show_individual, show_hand,
    show_ld, show_lp, show_rp, show_rd,
    show_overall, show_palmar_dorsal, show_measurement,
    show_diff, show_sex, show_pct_male_female, show_pct_diff, show_pct_lp_rp, show_pct_ld_rd
):
    if df.empty: return

    data = df.drop(columns="Time")
    if counts_per_sec:
        data, _ = _bin_to_cps(data)
        x = pd.Series(np.arange(len(data)), dtype=float)
        x_title, y_title = "Tijd (seconden)", "Detectie ratio (cps)"
        t_base, t_int, t_rec = PHASE_BASELINE_END, PHASE_INTERVENTION_END, PHASE_RECOVERY_END
    else:
        x = df["Time"]
        x_title, y_title = "Tijd index (50 ms)", "Detectie ratio (counts / 50 ms)"
        t_base, t_int, t_rec = PHASE_BASELINE_END * BIN_ROWS, PHASE_INTERVENTION_END * BIN_ROWS, PHASE_RECOVERY_END * BIN_ROWS

    smoothed = data.rolling(window=window, center=True, min_periods=1).mean()
    if demean: smoothed -= smoothed.mean()
    if start_at_zero: smoothed -= smoothed.iloc[0]

    C = px.colors.qualitative
    plot_config = [
        (show_all_individual, individual_labels, 1.0, C.Alphabet),
        (show_individual, file_labels, 1.5, C.Plotly),
        (show_hand, plot_groups.get('hand', []), 3.0, C.Plotly),
        (show_sex, plot_groups.get('sex', []), 4.0, C.Safe),
        (show_sex, plot_groups.get('sex_channel', []), 3.0, C.Safe),
        (show_pct_male_female, plot_groups.get('sex_diff', []), 3.5, ['#8c564b']),
        (show_ld, ['LD_Avg'], 3.0, [C.D3[0]]), (show_lp, ['LP_Avg'], 3.0, [C.D3[1]]),
        (show_rp, ['RP_Avg'], 3.0, [C.D3[2]]), (show_rd, ['RD_Avg'], 3.0, [C.D3[3]]),
        (show_overall, ['Overall_Avg'], 4.0, ['black']),
        (show_palmar_dorsal, ['Palmar_Avg (LP+RP)', 'Dorsal_Avg (LD+RD)'], 3.0, C.T10),
        (show_measurement, [f'm{i}_Avg' for i in range(1, 5)], 3.0, C.Vivid),
        (show_diff, ['Verschil (Palmair - Dorsaal)'], 3.5, ['#ff00ff']),
        (show_pct_diff, ['Procentueel_Verschil (%)'], 3.5, ['#00ffcc']),
        (show_pct_lp_rp, ['Procentueel_Verschil (LP vs RP) (%)'], 3.5, ['#e377c2']),
        (show_pct_ld_rd, ['Procentueel_Verschil (LD vs RD) (%)'], 3.5, ['#bcbd22']),
    ]

    fig = go.Figure()
    x_max = int(x.iloc[-1]) if not x.empty else 0
    
    fig.add_vrect(x0=0, x1=min(t_base, x_max), fillcolor="rgba(0, 255, 0, 0.05)", layer="below", line_width=0, annotation_text="Baseline (0-2m)", annotation_position="top left")
    if x_max > t_base: fig.add_vrect(x0=t_base, x1=min(t_int, x_max), fillcolor="rgba(255, 165, 0, 0.08)", layer="below", line_width=0, annotation_text="Interventie (2-5m)", annotation_position="top left")
    if x_max > t_int: fig.add_vrect(x0=t_int, x1=min(t_rec, x_max), fillcolor="rgba(0, 0, 255, 0.05)", layer="below", line_width=0, annotation_text="Herstel (5-10m)", annotation_position="top left")

    for show, col_labels, lw, palette in plot_config:
        if not show: continue
        for i, label in enumerate(col_labels):
            if label not in smoothed.columns: continue
            if label in individual_labels:
                opacity, width = 0.15, 1.0
            else:
                opacity, width = 0.4 if "pct" in label.lower() else 1.0, lw
            
            fig.add_trace(go.Scatter(
                x=x, y=smoothed[label], mode='lines', name=label,
                line=dict(color=palette[i % len(palette)], width=width), opacity=opacity,
                hovertemplate=(f"<b>{label}</b><br>Tijd: %{{x}}<br>Waarde: %{{y:.2f}}<extra></extra>"),
            ))

    fig.update_layout(
        title_text="<b>UPE Detectie & Fase Analyse</b>", title_font_size=22,
        xaxis_title=x_title, yaxis_title=y_title, plot_bgcolor='white', height=750, hovermode="x unified", font=dict(size=13),
        legend=dict(bgcolor='rgba(255,255,255,0.85)', bordercolor='lightgray', borderwidth=1, font_size=11),
        xaxis=dict(showgrid=False, linecolor='black', mirror=True), yaxis=dict(showgrid=True, gridcolor='lightgray', linecolor='black', mirror=True),
        margin=dict(t=60, b=60, l=70, r=20),
    )
    fig.show()

    # ==============================================================================
    # TABEL 1: Robuuste Gemiddelde Verschillen (Kanalen & Demografie)
    # ==============================================================================
    df_stats = smoothed.copy()
    df_stats['Fase'] = 'Onbekend'
    df_stats.loc[x < t_base, 'Fase'] = '1. Baseline (0-2m)'
    df_stats.loc[(x >= t_base) & (x < t_int), 'Fase'] = '2. Interventie (2-5m)'
    df_stats.loc[(x >= t_int) & (x <= t_rec), 'Fase'] = '3. Herstel (5-10m)'
    
    summary = df_stats.groupby('Fase').mean()
    
    html_str = f"""
    <div style="font-family: Arial, sans-serif; margin-top: 20px; padding: 15px; border: 1px solid #ddd; border-radius: 8px; background-color: #f9f9f9; max-width: 950px;">
        <h3 style="margin-top: 0; color: #333;">Gemiddelde Verschillen per Fase (Robuuste Methode)</h3>
        <p style="font-size: 13px; color: #666;">Berekent eerst het absolute gemiddelde over de hele fase, en berekent <i>daarna</i> pas de percentages. Inclusief Man/Vrouw (M/V) asymmetrie.</p>
        <table style="width: 100%; border-collapse: collapse; text-align: left;">
            <tr style="background-color: #eee; border-bottom: 2px solid #ccc;">
                <th style="padding: 8px;">Fase</th>
                <th style="padding: 8px;">Palmair - Dorsaal (Abs)</th>
                <th style="padding: 8px;">Palmair vs Dorsaal (%)</th>
                <th style="padding: 8px;">LP vs RP (%)</th>
                <th style="padding: 8px;">LD vs RD (%)</th>
                <th style="padding: 8px;">Man vs Vrouw (%)</th>
            </tr>
    """
    
    for index in summary.index:
        pal_val = summary.get('Palmar_Avg (LP+RP)', pd.Series(dtype=float)).get(index, np.nan)
        dor_val = summary.get('Dorsal_Avg (LD+RD)', pd.Series(dtype=float)).get(index, np.nan)
        lp_val, rp_val  = summary.get('LP_Avg', pd.Series(dtype=float)).get(index, np.nan), summary.get('RP_Avg', pd.Series(dtype=float)).get(index, np.nan)
        ld_val, rd_val  = summary.get('LD_Avg', pd.Series(dtype=float)).get(index, np.nan), summary.get('RD_Avg', pd.Series(dtype=float)).get(index, np.nan)
        male_val = summary.get('male_Avg', pd.Series(dtype=float)).get(index, np.nan)
        female_val = summary.get('female_Avg', pd.Series(dtype=float)).get(index,np.nan)

        abs_diff = np.nan if pd.isna(pal_val) else pal_val - dor_val
        pct_pal_dor = (pal_val - dor_val) / dor_val * 100 if dor_val else np.nan
        pct_lp_rp = (lp_val - rp_val) / rp_val * 100 if rp_val else np.nan
        pct_ld_rd = (ld_val - rd_val) / rd_val * 100 if rd_val else np.nan
        pct_mf = (male_val - female_val) / female_val * 100 if female_val else np.nan

        f_abs = f"{abs_diff:.3f}" if not pd.isna(abs_diff) else "-"
        f_pd  = f"{pct_pal_dor:.2f} %" if not pd.isna(pct_pal_dor) else "-"
        f_lprp = f"{pct_lp_rp:.2f} %" if not pd.isna(pct_lp_rp) else "-"
        f_ldrd = f"{pct_ld_rd:.2f} %" if not pd.isna(pct_ld_rd) else "-"
        f_mf = f"{pct_mf:.2f} %" if not pd.isna(pct_mf) else "-"

        html_str += f"""
            <tr style="border-bottom: 1px solid #ddd;">
                <td style="padding: 8px; font-weight: bold;">{index}</td>
                <td style="padding: 8px;">{f_abs}</td>
                <td style="padding: 8px;">{f_pd}</td>
                <td style="padding: 8px; color: #d62728;">{f_lprp}</td>
                <td style="padding: 8px; color: #7f7f7f;">{f_ldrd}</td>
                <td style="padding: 8px; color: #8c564b;">{f_mf}</td>
            </tr>
        """
    html_str += "</table></div>"
    display(DisplayHTML(html_str)) 

    # ==============================================================================
    # TABEL 2: Totale Extra Fotonen-emissie (Netto AUC)
    # ==============================================================================
    duration_int = PHASE_INTERVENTION_END - PHASE_BASELINE_END
    
    auc_results = []
    kanalen_om_te_testen = ['Palmar_Avg (LP+RP)', 'Dorsal_Avg (LD+RD)', 'LP_Avg', 'RP_Avg', 'LD_Avg', 'RD_Avg', 'male_Avg', 'female_Avg']
    
    for col in kanalen_om_te_testen:
        if col in smoothed.columns:
            baseline_val = smoothed.loc[x < t_base, col].mean()
            baseline_cps = baseline_val if counts_per_sec else baseline_val * BIN_ROWS
            
            verwacht_totaal = baseline_val * (t_int - t_base)
            actueel_totaal = smoothed.loc[(x >= t_base) & (x < t_int), col].sum()
            netto_extra = actueel_totaal - verwacht_totaal
            
            # Voorkom "feMannen" door de volgorde om te draaien (eerst female, dan male)
            label = col.replace('_Avg', '').replace('female', 'Vrouwen').replace('male', 'Mannen')
            
            auc_results.append({
                'Kanaal': label,
                'Baseline (CPS)': baseline_cps,
                'Verwacht Totaal': verwacht_totaal,
                'Actueel Totaal': actueel_totaal,
                'Netto Extra Fotonen (AUC)': netto_extra
            })

    df_auc = pd.DataFrame(auc_results)
    
    if not df_auc.empty:
        auc_html = f"""
        <div style="font-family: Arial, sans-serif; margin-top: 20px; margin-bottom: 30px; padding: 15px; border: 1px solid #ddd; border-radius: 8px; background-color: #f9f9f9; max-width: 950px;">
            <h3 style="margin-top: 0; color: #333;">Totale Extra Fotonen-emissie (Netto AUC) tijdens Interventie</h3>
            <p style="font-size: 13px; color: #666;">Dit toont hoeveel fotonen er in totaal <b>extra</b> zijn uitgezonden gedurende de 3 minuten van de interventie ten opzichte van de ruststaat. Bevat nu ook demografische data.</p>
            <table style="width: 100%; border-collapse: collapse; text-align: left;">
                <tr style="background-color: #eee; border-bottom: 2px solid #ccc;">
                    <th style="padding: 8px;">Kanaal / Demografie</th>
                    <th style="padding: 8px;">Normale Rust (CPS)</th>
                    <th style="padding: 8px;">Verwachte Som (180s)</th>
                    <th style="padding: 8px;">Gemeten Som (180s)</th>
                    <th style="padding: 8px; font-size: 14px; color: #d62728;"><b>Netto Extra Fotonen</b></th>
                </tr>
        """
        for _, row in df_auc.iterrows():
            auc_color = "#d62728" if row['Netto Extra Fotonen (AUC)'] > 0 else "#1f77b4"
            auc_html += f"""
                <tr style="border-bottom: 1px solid #ddd;">
                    <td style="padding: 8px; font-weight: bold;">{row['Kanaal']}</td>
                    <td style="padding: 8px;">{row['Baseline (CPS)']:.2f}</td>
                    <td style="padding: 8px;">{row['Verwacht Totaal']:.0f}</td>
                    <td style="padding: 8px;">{row['Actueel Totaal']:.0f}</td>
                    <td style="padding: 8px; font-weight: bold; color: {auc_color};">{row['Netto Extra Fotonen (AUC)']:.0f}</td>
                </tr>
            """
        auc_html += "</table></div>"
        display(DisplayHTML(auc_html))
        
        # OPMERKING: De fig_auc staafdiagram plot is hier op verzoek verwijderd!

# ---------------------------------------------------------
# UI helpers & Main
# ---------------------------------------------------------
def _section(title: str):
    return HTML(f"<div style='font-size:12px; font-weight:600; text-transform:uppercase; letter-spacing:0.06em; color:#666; margin:8px 0 2px;'>{title}</div>")

def _cb(desc: str, val: bool = False):
    return Checkbox(value=val, description=desc, style={'description_width': 'initial'}, layout=Layout(width='280px'))

_wide = Layout(width='420px')

processed_df, file_labels, plot_groups, individual_labels = load_and_process_data(DATA_FOLDER)

_n_bg = sum('_BG_' in c for c in individual_labels)
if _n_bg: print(f"✓ Loaded {_n_bg} background channel(s).")
if file_labels: print(f"✓ Loaded {len(file_labels)} hand-measurement file(s).")

slider = IntSlider(min=1, max=1001, step=10, value=101, description="Window:", style={'description_width': '60px'}, layout=_wide)

counts_per_sec_cb = _cb("Counts per second (cps)", val=False)
demean_cb = _cb("Demean recordings")
start_at_zero_cb = _cb("Start all at y = 0")

show_all_indiv_cb = _cb("All individual recordings", val=False)
show_indiv_cb = _cb("Per-file averages")
show_hand_cb = _cb("Per-hand averages")
show_pal_dors_cb = _cb("Palmar & dorsal averages", val=True)
show_meas_cb = _cb("Per-measurement averages")
show_overall_cb = _cb("Overall average")

show_diff_cb = _cb("Toon Verschil (Palmair - Dorsaal)", val=True)
show_pct_diff_cb = _cb("Toon Procentueel Verschil (%)", val=False)

show_ld_cb = _cb("Toon LD (Links Dorsaal)", val=False)
show_lp_cb = _cb("Toon LP (Links Palmair)", val=False)
show_rp_cb = _cb("Toon RP (Rechts Palmair)", val=False)
show_rd_cb = _cb("Toon RD (Rechts Dorsaal)", val=False)

show_pct_lp_rp_cb = _cb("Toon % Verschil (LP vs RP)", val=False)
show_pct_ld_rd_cb = _cb("Toon % Verschil (LD vs RD)", val=False)

show_sex_cb = _cb("Toon mannen vs vrouwen", val= True)
show_pct_mf_cb = _cb("Toon % verschil (M vs V)", val= False)

ui = VBox([
    HTML("<h3 style='margin:0 0 8px; font-size:15px;'>UPE Analyse Master Dashboard</h3>"),
    _section("Smoothing"), HBox([slider]),
    _section("Tijd / Schaal"), HBox([counts_per_sec_cb, demean_cb, start_at_zero_cb]),
    _section("Analyse & Verschillen (Vlakken)"),
    HBox([show_pal_dors_cb, show_diff_cb, show_pct_diff_cb]),
    HBox([show_pct_lp_rp_cb, show_pct_ld_rd_cb]),
    HBox([show_sex_cb, show_pct_mf_cb]),
    _section("Losse Kanalen (Gemiddelden)"),
    HBox([show_ld_cb, show_lp_cb, show_rp_cb, show_rd_cb]),
    _section("Overige Traces"),
    HBox([show_all_indiv_cb, show_indiv_cb, show_hand_cb]),
    HBox([show_meas_cb, show_overall_cb]),
    HTML("<div style='height:8px'></div>"),
], layout=Layout(padding='12px', border='1px solid #ddd', border_radius='8px', width='fit-content'))

out = interactive_output(plot_combined_data, {
    'df': fixed(processed_df), 'file_labels': fixed(file_labels),
    'plot_groups': fixed(plot_groups), 'individual_labels': fixed(individual_labels),
    'window': slider, 'counts_per_sec': counts_per_sec_cb,
    'demean': demean_cb, 'start_at_zero': start_at_zero_cb,
    'show_all_individual': show_all_indiv_cb, 'show_individual': show_indiv_cb,
    'show_hand': show_hand_cb, 'show_ld': show_ld_cb, 'show_lp': show_lp_cb,
    'show_rp': show_rp_cb, 'show_rd': show_rd_cb, 'show_overall': show_overall_cb,
    'show_palmar_dorsal': show_pal_dors_cb, 'show_measurement': show_meas_cb,
    'show_diff': show_diff_cb, 'show_pct_diff': show_pct_diff_cb,
    'show_pct_lp_rp': show_pct_lp_rp_cb, 'show_pct_ld_rd': show_pct_ld_rd_cb,
    'show_sex': show_sex_cb, 'show_pct_male_female': show_pct_mf_cb,
})

if not processed_df.empty:
    display(ui, out)
else:
    print("Execution halted: no data loaded.")

✓ Loaded 38 hand-measurement file(s).


Output()

In [7]:
# ===============================================================================
# Hier is de code voor de plots voor PALMAIR VS DORSAAL. Staafdiagram en grafiek
# ===============================================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import numpy as np
import pandas as pd

# 1. Zet data om naar Seconden (CPS) voor strakke publicatie-grafieken
try:
    poster_data, _ = _bin_to_cps(processed_df.drop(columns=["Time"]))
    x_sec = np.arange(len(poster_data))
except NameError:
    print("Fout: Run eerst het hoofdscript zodat de data is ingeladen!")

# Lichte smoothing voor de poster (bijv. 5 seconden window)
window_sec = 5
poster_smoothed = poster_data.rolling(window=window_sec, center=True, min_periods=1).mean()

# Fases definiëren
fases = [
    {"name": "Baseline", "start": 0, "end": 120, "color": "rgba(0, 255, 0, 0.05)"},
    {"name": "Interventie", "start": 120, "end": 300, "color": "rgba(255, 165, 0, 0.08)"},
    {"name": "Herstel", "start": 300, "end": 600, "color": "rgba(0, 0, 255, 0.05)"}
]

poster_layout = dict(
    plot_bgcolor='white',
    font=dict(family="Arial", size=18, color="black"), # Grote font voor poster
    xaxis=dict(showgrid=False, linecolor='black', linewidth=2, mirror=True, title="Tijd (s)"),
    yaxis=dict(showgrid=True, gridcolor='lightgray', linecolor='black', linewidth=2, mirror=True, title="UPE (Counts per seconde)"),
    margin=dict(t=80, b=80, l=80, r=40),
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.9)', bordercolor='black', borderwidth=1)
)

def add_phase_backgrounds(fig, max_x):
    for f in fases:
        if max_x > f['start']:
            fig.add_vrect(x0=f['start'], x1=min(f['end'], max_x), fillcolor=f['color'], 
                          layer="below", line_width=0, annotation_text=f['name'], 
                          annotation_position="top left", annotation_font_size=16)

max_x = int(x_sec[-1]) if len(x_sec) > 0 else 600

# ==============================================================================
# PLOT 1: (Totale Gemiddelde)
# ==============================================================================
fig1 = go.Figure()
if 'Palmar_Avg (LP+RP)' in poster_smoothed.columns and 'Dorsal_Avg (LD+RD)' in poster_smoothed.columns:
    fig1.add_trace(go.Scatter(x=x_sec, y=poster_smoothed['Palmar_Avg (LP+RP)'], name='Palmair Gemiddelde', line=dict(color='#1f77b4', width=4)))
    fig1.add_trace(go.Scatter(x=x_sec, y=poster_smoothed['Dorsal_Avg (LD+RD)'], name='Dorsaal Gemiddelde', line=dict(color='#7f7f7f', width=4)))
    
    add_phase_backgrounds(fig1, max_x)
    
    fig1.update_layout(**poster_layout)
    
    # Hier is de legenda specifiek voor Plot 1 naar rechts-midden verplaatst
    fig1.update_layout(
        title="<b>Tijdlijn: Gemiddelde UPE Detectie (Palmair vs Dorsaal)</b>", 
        title_font_size=24,
        legend=dict(x=0.98, y=0.5, xanchor="right", yanchor="middle")
    )
    
    fig1.show()

# ==============================================================================
# PLOT 2: Staafdiagram met Fase-Gemiddeldes
# ==============================================================================
if 'Palmar_Avg (LP+RP)' in poster_smoothed.columns:
    df_fase = poster_smoothed[['Palmar_Avg (LP+RP)', 'Dorsal_Avg (LD+RD)']].copy()
    df_fase['Fase'] = pd.cut(x_sec, bins=[-1, 120, 300, 600], labels=['Baseline', 'Interventie', 'Herstel'])
    
    mean_data = df_fase.groupby('Fase').mean()
    std_data = df_fase.groupby('Fase').std() # Varianties binnen de fase
    
    fig2 = go.Figure()
    fig2.add_trace(go.Bar(name='Palmair', x=mean_data.index, y=mean_data['Palmar_Avg (LP+RP)'], marker_color='#1f77b4',
                          error_y=dict(type='data', array=std_data['Palmar_Avg (LP+RP)'], visible=True)))
    fig2.add_trace(go.Bar(name='Dorsaal', x=mean_data.index, y=mean_data['Dorsal_Avg (LD+RD)'], marker_color='#7f7f7f',
                          error_y=dict(type='data', array=std_data['Dorsal_Avg (LD+RD)'], visible=True)))
    
    fig2.update_layout(**poster_layout)
    
    # Hier is de legenda overschreven specifiek voor dit figuur
    fig2.update_layout(
        barmode='group', 
        title="<b>Gemiddelde Fotonenemissie per Experimentele Fase</b>", 
        title_font_size=24, 
        xaxis_title="Fase", 
        yaxis_title="Gemiddelde UPE (CPS)",
        legend=dict(x=0.98, y=0.98, xanchor="right", yanchor="top") 
    )
    fig2.show()



In [8]:
# ===============================================================================================
# Berekent de netto extra fotonen-emissie tijdens de interventie door opp van de grafiek te nemen
# ===============================================================================================

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, HTML

# 1. Zorg dat we met Count Per Second (CPS) data werken
try:
    auc_data, _ = _bin_to_cps(processed_df.drop(columns=["Time"]))
    t_sec = np.arange(len(auc_data))
except NameError:
    print("Fout: Run eerst het hoofdscript zodat de data is ingeladen!")

# Tijdsframes (in seconden)
t_base_end = 120
t_int_end = 300
duration_int = t_int_end - t_base_end # 180 seconden

kanalen_om_te_testen = [
    'Palmar_Avg (LP+RP)', 'Dorsal_Avg (LD+RD)', 
    'LP_Avg', 'RP_Avg', 'LD_Avg', 'RD_Avg'
]

auc_results = []

# 2. Bereken de AUC per kanaal
for col in kanalen_om_te_testen:
    if col in auc_data.columns:
        # Stap A: Gemiddelde Baseline per seconde
        baseline_cps = auc_data.loc[t_sec < t_base_end, col].mean()
        
        # Stap B: Verwachte totale emissie over 3 minuten ZONDER effect
        verwacht_totaal = baseline_cps * duration_int
        
        # Stap C: Daadwerkelijke totale emissie (De echte 'Area under the Curve')
        actueel_totaal = auc_data.loc[(t_sec >= t_base_end) & (t_sec < t_int_end), col].sum()
        
        # Stap D: Netto effect (Hoeveel EXTRA fotonen zijn er geproduceerd?)
        netto_extra = actueel_totaal - verwacht_totaal
        
        auc_results.append({
            'Kanaal': col.replace('_Avg', ''),
            'Baseline (CPS)': baseline_cps,
            'Verwacht Totaal': verwacht_totaal,
            'Actueel Totaal': actueel_totaal,
            'Netto Extra Fotonen (AUC)': netto_extra
        })

df_auc = pd.DataFrame(auc_results)

# ==============================================================================
# WEERGAVE 1: De Statistische Tabel
# ==============================================================================
html_str = f"""
<div style="font-family: Arial, sans-serif; margin-top: 20px; margin-bottom: 30px; padding: 15px; border: 1px solid #ddd; border-radius: 8px; background-color: #f9f9f9; max-width: 900px;">
    <h3 style="margin-top: 0; color: #333;">Totale Extra Fotonen-emissie (Netto AUC) tijdens Interventie</h3>
    <p style="font-size: 13px; color: #666;">Dit toont hoeveel fotonen er in totaal <b>extra</b> zijn uitgezonden gedurende de volle 3 minuten (180s) van de interventie ten opzichte van het verwachte baseline-metabolisme.</p>
    <table style="width: 100%; border-collapse: collapse; text-align: left;">
        <tr style="background-color: #eee; border-bottom: 2px solid #ccc;">
            <th style="padding: 8px;">Kanaal</th>
            <th style="padding: 8px;">Normale Rust (CPS)</th>
            <th style="padding: 8px;">Verwachte Som (180s)</th>
            <th style="padding: 8px;">Gemeten Som (180s)</th>
            <th style="padding: 8px; font-size: 14px; color: #d62728;"><b>Netto Extra Fotonen</b></th>
        </tr>
"""
for _, row in df_auc.iterrows():
    # Kleur rood als het gestegen is, blauw als de emissie is gedaald
    auc_color = "#d62728" if row['Netto Extra Fotonen (AUC)'] > 0 else "#1f77b4"
    html_str += f"""
        <tr style="border-bottom: 1px solid #ddd;">
            <td style="padding: 8px; font-weight: bold;">{row['Kanaal']}</td>
            <td style="padding: 8px;">{row['Baseline (CPS)']:.2f}</td>
            <td style="padding: 8px;">{row['Verwacht Totaal']:.0f}</td>
            <td style="padding: 8px;">{row['Actueel Totaal']:.0f}</td>
            <td style="padding: 8px; font-weight: bold; color: {auc_color};">{row['Netto Extra Fotonen (AUC)']:.0f}</td>
        </tr>
    """
html_str += "</table></div>"
display(HTML(html_str))


Kanaal,Normale Rust (CPS),Verwachte Som (180s),Gemeten Som (180s),Netto Extra Fotonen
Palmar (LP+RP),34.22,6160,6051,-109
Dorsal (LD+RD),27.86,5015,4713,-302
LP,36.55,6579,6550,-29
RP,31.90,5741,5552,-189
LD,25.55,4600,4295,-305
RD,30.17,5430,5132,-298
